In [12]:
import cv2
import numpy as np
from ultralytics import YOLO
from collections import deque
import datetime
import uuid
from google.colab.patches import cv2_imshow

model = YOLO("yolov8n.pt")
VEHICLE_CLASSES = [2, 5, 7]

KNOWN_WIDTHS = {
    2: 1.8,
    5: 2.5,
    7: 2.5
}

FOCAL_LENGTH = 750.0

class SimpleLaneDetector:
    def __init__(
        self,
        departure_threshold=0.28,
        approach_threshold=0.10,
        fast_change_frames=5,
        weave_window=10,
        weave_min_crossings=2,
        weave_cooldown_frames=15,
    ):
        self.departure_threshold = departure_threshold
        self.approach_threshold = approach_threshold
        self.fast_change_frames = fast_change_frames
        self.weave_min_crossings = weave_min_crossings
        self.weave_cooldown_frames = weave_cooldown_frames

        self.transit_frames = 0
        self.currently_departed = False
        self.missing_frames = 0

        self.weave_history = deque(maxlen=weave_window)
        self.frames_since_weave_alert = weave_cooldown_frames

    def reset(self):
        self.transit_frames = 0
        self.currently_departed = False
        self.missing_frames = 0
        self.weave_history.clear()
        self.frames_since_weave_alert = self.weave_cooldown_frames

    def _find_lane_lines(self, frame):
        height, width = frame.shape[:2]
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        blur = cv2.GaussianBlur(gray, (5, 5), 0)
        edges = cv2.Canny(blur, 50, 150)

        mask = np.zeros_like(edges)
        polygon = np.array([[
            (int(width * 0.1), height),
            (int(width * 0.45), int(height * 0.6)),
            (int(width * 0.55), int(height * 0.6)),
            (int(width * 0.9), height),
        ]], np.int32)
        cv2.fillPoly(mask, polygon, 255)
        masked_edges = cv2.bitwise_and(edges, mask)

        lines = cv2.HoughLinesP(
            masked_edges, 1, np.pi / 180,
            threshold=40, minLineLength=50, maxLineGap=150,
        )

        left_points = []
        right_points = []

        if lines is not None:
            for line in lines:
                x1, y1, x2, y2 = line[0]
                if x2 == x1:
                    continue
                slope = (y2 - y1) / (x2 - x1)
                if slope < -0.5:
                    left_points.extend([x1, x2])
                elif slope > 0.5:
                    right_points.extend([x1, x2])

        return left_points, right_points, width, height

    def _compute_offset(self, frame):
        left_points, right_points, width, height = self._find_lane_lines(frame)
        left_x = float(np.mean(left_points)) if left_points else None
        right_x = float(np.mean(right_points)) if right_points else None

        if left_x is None and right_x is None:
            return None

        frame_center = width / 2.0
        if left_x is not None and right_x is not None:
            lane_center = (left_x + right_x) / 2.0
        elif left_x is not None:
            lane_center = left_x + (width * 0.25)
        else:
            lane_center = right_x - (width * 0.25)

        offset = (lane_center - frame_center) / (width / 2.0)
        return max(-1.0, min(1.0, offset))

    def process_frame(self, frame):
        offset = self._compute_offset(frame)
        self.frames_since_weave_alert += 1

        if offset is None:
            self.missing_frames += 1
            if self.missing_frames == 8:
                return {
                    "event_type": "lane_departure",
                    "detected": False,
                    "reason": "lane_markings_not_clear",
                    "note": "Lane markings are not clear",
                }
            return None

        self.missing_frames = 0
        self.weave_history.append(offset)

        weave_event = self._check_weaving()
        if weave_event:
            return weave_event

        return self._check_departure(offset)

    def _check_weaving(self):
        if len(self.weave_history) < self.weave_history.maxlen:
            return None
        if self.frames_since_weave_alert < self.weave_cooldown_frames:
            return None

        direction_changes = 0
        last_sign = 0
        values = list(self.weave_history)
        for i in range(1, len(values)):
            delta = values[i] - values[i - 1]
            if abs(delta) < 0.05:
                continue
            sign = 1 if delta > 0 else -1
            if last_sign != 0 and sign != last_sign:
                direction_changes += 1
            last_sign = sign

        if direction_changes >= self.weave_min_crossings:
            self.frames_since_weave_alert = 0
            return {
                "event_type": "fatigue",
                "detected": True,
                "pattern": "weaving",
                "likely_cause": "drowsiness",
                "severity": "high",
                "note": "Possible Drowsiness (Weaving detected)",
            }
        return None

    def _check_departure(self, offset):
        abs_offset = abs(offset)
        if abs_offset >= self.departure_threshold:
            if self.currently_departed:
                return None

            self.currently_departed = True
            frames_it_took = self.transit_frames
            self.transit_frames = 0

            if frames_it_took <= self.fast_change_frames:
                return None

            direction = "right" if offset > 0 else "left"
            return {
                "event_type": "lane_departure",
                "detected": True,
                "pattern": "gradual_drift",
                "likely_cause": "distraction_or_drowsiness",
                "severity": "medium",
                "direction": direction,
                "note": f"Gradual drift to {direction} (distraction/drowsiness)",
            }

        elif abs_offset >= self.approach_threshold:
            self.transit_frames += 1
            self.currently_departed = False
            return None
        else:
            self.transit_frames = 0
            self.currently_departed = False
            return None


class ProductionSafeDistanceTracker:
    def __init__(self, safe_distance_m=15.0, cooldown_frames=15):
        self.safe_distance_m = safe_distance_m
        self.cooldown_frames = cooldown_frames
        self.frames_since_alert = cooldown_frames
        self.dist_history = deque(maxlen=6)

    def reset(self):
        self.dist_history.clear()
        self.frames_since_alert = self.cooldown_frames

    def process(self, cls_id, box_pixel_width):
        self.frames_since_alert += 1
        real_w = KNOWN_WIDTHS.get(cls_id, 1.8)
        current_dist = (real_w * FOCAL_LENGTH) / max(box_pixel_width, 1)
        self.dist_history.append(current_dist)

        if len(self.dist_history) < self.dist_history.maxlen:
            return current_dist, None

        approach_speed = (self.dist_history[0] - self.dist_history[-1]) / len(self.dist_history)
        is_too_close = current_dist < self.safe_distance_m
        is_closing_fast = approach_speed > 0.4

        if (is_too_close or is_closing_fast) and (self.frames_since_alert >= self.cooldown_frames):
            self.frames_since_alert = 0
            severity = "high" if (current_dist < 8.0 or (is_too_close and is_closing_fast)) else "medium"
            note = f"Very close with high approach speed ({current_dist:.1f}m)" if (is_too_close and is_closing_fast) else (f"Unsafe tailgating distance ({current_dist:.1f}m)" if is_too_close else f"Rapidly approaching vehicle ahead ({current_dist:.1f}m)")

            return current_dist, {
                "event_type": "unsafe_distance",
                "distance_m": round(current_dist, 1),
                "severity": severity,
                "note": note
            }

        return current_dist, None


SEVERITY_MAP = {
    "critical": "high",
    "high": "high",
    "warning": "medium",
    "medium": "medium",
    "info": "low",
    "low": "low"
}

def log_violation_event(event_data, trip_id="T001", location=None):
    if location is None:
        location = {"lat": 24.264, "lng": 46.3519}

    unique_id = f"E_{trip_id}_{str(uuid.uuid4())[:6]}"
    utc_timestamp = datetime.datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
    raw_severity = event_data.get("severity", "medium")
    severity = SEVERITY_MAP.get(raw_severity, "medium")

    event_payload = {
        "event_id": unique_id,
        "type": event_data.get("event_type", "safety_violation"),
        "timestamp": utc_timestamp,
        "severity": severity,
        "details": event_data.get("note", "Safety infraction detected"),
        "location": location
    }
    print(f"\n🚨 [LOGGED TO DB] >>> {event_payload}\n")
    return event_payload


def run_unified_safety_system(video_source=0):
    cap = cv2.VideoCapture(video_source)

    lane_detector = SimpleLaneDetector()
    distance_tracker = ProductionSafeDistanceTracker(safe_distance_m=15.0)

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        h, w, _ = frame.shape

        lane_alert = lane_detector.process_frame(frame)
        if lane_alert and lane_alert.get("detected"):
            log_violation_event(lane_alert, trip_id="T001")
            alert_text = f"LANE: {lane_alert['note']}"
            cv2.putText(frame, alert_text, (30, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

        results = model(frame, stream=True, verbose=False)
        closest_target = None
        min_distance = 999.0

        for r in results:
            for box in r.boxes:
                cls_id = int(box.cls[0])
                if cls_id in VEHICLE_CLASSES:
                    x1, y1, x2, y2 = map(int, box.xyxy[0])
                    box_w = x2 - x1
                    box_center_x = (x1 + x2) / 2

                    if (w * 0.35) < box_center_x < (w * 0.65):
                        real_w = KNOWN_WIDTHS.get(cls_id, 1.8)
                        approx_dist = (real_w * FOCAL_LENGTH) / max(box_w, 1)

                        if approx_dist < min_distance:
                            min_distance = approx_dist
                            closest_target = (cls_id, box_w, (x1, y1, x2, y2))

        if closest_target is not None:
            cls_id, box_w, (x1, y1, x2, y2) = closest_target
            dist_m, dist_alert = distance_tracker.process(cls_id, box_w)

            color = (0, 0, 255) if dist_m < 15.0 else (0, 255, 0)
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, f"Dist: {dist_m:.1f}m", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

            if dist_alert:
                log_violation_event(dist_alert, trip_id="T001")
                cv2.putText(frame, f"DIST: {dist_alert['note']}", (30, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
        else:
            distance_tracker.reset()

        cv2_imshow(frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()


if __name__ == "__main__":
    run_unified_safety_system(0)